# NEWS CATEGORY CLASSIFICATION

#### DATASET DOWNLOADING

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-category-dataset")

print("Path to dataset files:", path)

p:\PROJECTS\DATA SCIENCE\NEWS CATEGORY CLASSIFICATION\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\khage\.cache\kagglehub\datasets\rmisra\news-category-dataset\versions\3


#### DATASET LOADING

In [2]:
import pandas as pd
import json

path+="/News_Category_Dataset_v3.json"
df = pd.read_json(path, lines=True)

#### DATA INSPECTION

GETTING SHAPE OF THE DATASET

In [3]:
df.shape

(209527, 6)

DISPLAYING FIRST 5 RECORDS


In [4]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


GETTING HOW MANY RECORDS ARE THERE IN EACH CATEGORY

In [5]:
print(df['category'].value_counts())

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

GETTING TOTAL RECORDS AND CATEGORY COUNTS

In [6]:
print(f"Total records: {len(df)}")
print(f"Categories: {df['category'].nunique()}")

Total records: 209527
Categories: 42


#### DATA CLEANING

CHECKING NULL OCCURANCES

In [7]:
df.isnull().sum()

link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
dtype: int64

CHECKING DUPLICATE VALUES

In [8]:
print(df.duplicated().sum())
dupRow = df[df.duplicated()]
print(dupRow)

13
                                                     link  \
67677   https://www.huffingtonpost.comhttp://www.mothe...   
67923   https://www.huffingtonpost.comhttp://gizmodo.c...   
70239   https://www.huffingtonpost.comhttp://www.cnbc....   
139830  https://www.huffingtonpost.comhttp://www.cnn.c...   
144409  https://www.huffingtonpost.comhttp://www.upwor...   
145142  https://www.huffingtonpost.comhttp://www.weath...   
178155  https://www.huffingtonpost.comhttp://www.busin...   
187329  https://www.huffingtonpost.comhttp://www.nytim...   
194596  https://www.huffingtonpost.comhttp://blogs.wsj...   
194598  https://www.huffingtonpost.comhttp://www.theda...   
207122  https://www.huffingtonpost.comhttp://d.repubbl...   
207208  https://www.huffingtonpost.comhttp://d.repubbl...   
207318  https://www.huffingtonpost.comhttp://d.repubbl...   

                                                 headline        category  \
67677   On Facebook, Trump's Longtime Butler Calls For...        

REMOVING DUPLICATE VALUES

In [9]:
df = df.drop_duplicates()

CREATING NEW COLUMN AS "text" FOR BOTH HEADLINE AND DESCRIPTION

In [10]:
df['text'] = df['headline'] + ' ' + df['short_description']

REMOVING EMPTY TEXT

In [11]:
df = df[df['text'].str.strip() != ""]

REMOVING COLUMS THAT ARE NOT REQUIRED

In [12]:
df.drop(columns=['link','short_description','headline','authors'], inplace=True)

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["label"] = le.fit_transform(df["category"])

num_labels = len(le.classes_)


In [14]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

#### TOKENIZATION

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from transformers import DataCollatorWithPadding
from transformers import TrainerCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer


In [16]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

In [ ]:
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# =====================================================
# TOKENIZATION
# =====================================================

def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

# =====================================================
# DYNAMIC PADDING
# =====================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# =====================================================
# CLASS WEIGHTS
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("\nClass Weights:")
print(class_weights)

# =====================================================
# LOAD MODEL
# =====================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# =====================================================
# METRICS
# =====================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

# =====================================================
# CUSTOM TRAINER
# =====================================================

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_function = nn.CrossEntropyLoss(
            weight=class_weights.to(
                model.device
            )
        )

        loss = loss_function(
            logits.view(
                -1,
                model.config.num_labels
            ),
            labels.view(-1)
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

# =====================================================
# TRAINING CONFIG
# =====================================================

training_args = TrainingArguments(

    output_dir="./news_classifier",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    warmup_ratio=0.1,

    lr_scheduler_type="cosine",

    weight_decay=0.01,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=6,

    fp16=True,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    greater_is_better=True,

    logging_steps=100,

    report_to="none"
)

# =====================================================
# TRAINER
# =====================================================

trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# =====================================================
# TRAIN
# =====================================================

trainer.train()

# =====================================================
# FINAL EVALUATION
# =====================================================

results = trainer.evaluate()

print("\nFinal Results")
print(results)


p:\PROJECTS\DATA SCIENCE\NEWS CATEGORY CLASSIFICATION\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khage\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 41902/41902 [00:01<00:00, 22235.78 examples/s]



Class Weights:
tensor([3.3062, 3.7261, 1.0886, 0.8326, 4.3614, 0.9238, 1.4002, 4.6457, 1.4559,
        4.9206, 0.2873, 3.4581, 3.5599, 0.7868, 3.5694, 1.9021, 0.7452, 1.1547,
        1.4319, 4.4144, 1.6953, 2.8403, 0.5674, 1.2613, 0.1401, 0.7860, 1.9353,
        2.2610, 0.9824, 2.2133, 0.5084, 2.3796, 2.3754, 1.3615, 0.5039, 3.6213,
        1.3657, 1.7960, 0.2780, 1.3968, 1.5122, 1.9353])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3095.50it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.341954,1.318067,0.604410,0.545275
2,1.006909,1.223490,0.658632,0.596261
3,0.783368,1.259860,0.663668,0.597125
4,0.411413,1.418128,0.679466,0.612970
5,0.334578,1.595243,0.690635,0.617250
6,0.252423,1.648216,0.694334,0.619101


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.56it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.252423,1.648216,6,0.694334,0.619101



Final Results
{'eval_loss': 1.6482160091400146, 'eval_accuracy': 0.6943343993126819, 'eval_f1_macro': 0.6191011359058379}


: 